In [1]:
from google.colab import drive
drive.mount('/content/drive')  # Mount Google Drive

Mounted at /content/drive


In [2]:
import os
import numpy as np
import cv2
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Define paths to drought and non-drought folders
drought_path = '/content/drive/My Drive/Dataset/Drought/'
nondrought_path = '/content/drive/My Drive/Dataset/Nondrought/'

# Resize images to 128x128
def load_and_preprocess_images(image_folder, size=(128, 128)):
    images = []
    labels = []

    for label, folder in enumerate([drought_path, nondrought_path]):
        folder_path = drought_path if label == 0 else nondrought_path
        for image_name in os.listdir(folder_path):
            img = cv2.imread(os.path.join(folder_path, image_name))
            img_resized = cv2.resize(img, size)
            img_normalized = img_resized / 255.0  # Normalize to [0,1]
            images.append(img_normalized)
            labels.append(label)

    return np.array(images), np.array(labels)

# Load and preprocess images
images, labels = load_and_preprocess_images(drought_path)
print(f'Loaded {images.shape[0]} images')

Loaded 200 images


In [10]:
def color_histograms(images, bins=16):
    hist_features = []
    for img in images:
        # Check if image is grayscale and convert it to BGR
        if len(img.shape) == 2:  # If grayscale
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        elif len(img.shape) != 3:
            print("Error: Image does not have 3 channels.")
            continue

        # Check if image is in valid format
        print(f"Processing image with shape: {img.shape}")

        # Calculate histogram for each channel (R, G, B)
        hist = cv2.calcHist([img], [0, 1, 2], None, [bins, bins, bins], [0, 256, 0, 256, 0, 256])
        hist = hist.flatten()  # Flatten the histogram to 1D
        hist_features.append(hist)

    return np.array(hist_features)

In [11]:
# Function to calculate color statistics (mean, standard deviation for each channel)
def color_statistics(images):
    means = np.mean(images, axis=(1, 2))  # Mean across width and height
    stds = np.std(images, axis=(1, 2))    # Standard deviation across width and height
    return np.hstack([means, stds])  # Combine mean and std into a single vector

# Extract color statistics from images
color_stats = color_statistics(images)
print(f'Color statistics extracted for {color_stats.shape[0]} images')

Color statistics extracted for 200 images


In [14]:
# Function to calculate color histograms (RGB or HSV)
def color_histograms(images, bins=16):
    hist_features = []
    for img in images:
        # Check if image is grayscale and convert it to BGR
        if len(img.shape) == 2:  # If grayscale
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        # Ensure the image is of type np.uint8 (8-bit unsigned integer)
        if img.dtype != np.uint8:
            print(f"Converting image dtype from {img.dtype} to np.uint8")
            img = np.uint8(img)

        # Calculate histogram for each channel (R, G, B)
        hist = cv2.calcHist([img], [0, 1, 2], None, [bins, bins, bins], [0, 256, 0, 256, 0, 256])
        hist = hist.flatten()  # Flatten the histogram to 1D
        hist_features.append(hist)

    return np.array(hist_features)

# Extract color histograms from images
color_hist = color_histograms(images)
print(f'Color histograms extracted for {color_hist.shape[0]} images')

Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to np.uint8
Converting image dtype from float64 to n

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(color_hist, labels, test_size=0.3, random_state=42)

# Linear SVM (baseline)
linear_svm = SVC(kernel='linear')
linear_svm.fit(X_train, y_train)

# RBF SVM (Non-linear)
rbf_svm = SVC(kernel='rbf')
rbf_svm.fit(X_train, y_train)

# Evaluate the models
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f'Classification Report:\n{classification_report(y_test, y_pred)}')
    print(f'Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}')

# Evaluate Linear SVM
print("Linear SVM Evaluation:")
evaluate_model(linear_svm, X_test, y_test)

# Evaluate RBF SVM
print("RBF SVM Evaluation:")
evaluate_model(rbf_svm, X_test, y_test)

Linear SVM Evaluation:
Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        31
           1       0.48      1.00      0.65        29

    accuracy                           0.48        60
   macro avg       0.24      0.50      0.33        60
weighted avg       0.23      0.48      0.31        60

Confusion Matrix:
[[ 0 31]
 [ 0 29]]
RBF SVM Evaluation:
Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        31
           1       0.48      1.00      0.65        29

    accuracy                           0.48        60
   macro avg       0.24      0.50      0.33        60
weighted avg       0.23      0.48      0.31        60

Confusion Matrix:
[[ 0 31]
 [ 0 29]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m